# 04. CIES Experiment: Credibility Index via Explanation Stability

## Tổng quan thí nghiệm CIES
Thí nghiệm đo lường độ ổn định của giải thích SHAP qua các lần bootstrap resample tập huấn luyện.

### Thuật toán bắt buộc (spec mục 7.1):
Với mỗi tổ hợp (Model × Imbalance Technique):
```
Cố định tập df_test_fixed_eval (đánh giá trên cùng mẫu eval)
Lặp N_RUNS lần (seed = 0 ... N_RUNS - 1):
  1. Bootstrap resample tập train
  2. Fit lại encoding từ đầu trên resample (KHÔNG dùng encoding cũ)
  3. Áp dụng kỹ thuật imbalance
  4. Huấn luyện mô hình
  5. Tính SHAP values trên df_test_fixed_eval
Tính CIES Score = 1 - Mean(Rank-Weighted Distance)
```

### Rank-Weighted Distance:
Trọng số $w_r = 1/r$ phạt nặng hơn khi các top features quan trọng bị hoán đổi thứ hạng.

In [ ]:
# 1. Setup & Imports
import sys
from pathlib import Path
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_DATA_DIR, RESULTS_DIR, SEED, N_RUNS, TARGET_COL,
    MODEL_NAMES, IMBALANCE_TECHNIQUES, ONEHOT_COLS, TARGET_ENCODE_COLS
)
from src.explainability.cies import run_cies_experiment_isolated, save_cies_results

print(f'SEED: {SEED}, N_RUNS config: {N_RUNS}')
print('Models:', MODEL_NAMES)
print('Imbalance techniques:', IMBALANCE_TECHNIQUES)

In [ ]:
# 2. Chuẩn bị tập Train & Fixed Eval Set (RAW — chưa encode)
# QUAN TRỌNG: phải đọc bản RAW (train_raw.parquet / test_raw.parquet),
# KHÔNG phải *_encoded.parquet — vì run_cies_experiment() cần tự fit lại
# encoding từ đầu trên mỗi bootstrap resample ở mỗi run (spec mục 7.1).
# Nếu đọc *_encoded.parquet, các cột categorical gốc đã bị drop và bước
# re-encode bên trong sẽ thành no-op câm (dùng lại encoding cũ).
train_raw = pd.read_parquet(Path('../data/processed/train_raw.parquet'))
test_raw = pd.read_parquet(Path('../data/processed/test_raw.parquet'))

# Để đảm bảo tốc độ tính SHAP lặp lại N lần, lấy mẫu cố định eval set đại diện
# Bao gồm cả fraud và non-fraud
eval_fraud = test_raw[test_raw[TARGET_COL] == 1].sample(n=min(50, test_raw[TARGET_COL].sum()), random_state=SEED)
eval_legit = test_raw[test_raw[TARGET_COL] == 0].sample(n=min(200, (test_raw[TARGET_COL] == 0).sum()), random_state=SEED)
df_test_fixed_eval = pd.concat([eval_fraud, eval_legit]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f'Fixed Eval Set: {len(df_test_fixed_eval)} mẫu (Fraud: {df_test_fixed_eval[TARGET_COL].sum()})')
print('Eval set này sẽ được GIỮ NGUYÊN qua toàn bộ N_RUNS.')

--- 
## 3. Chạy Thí Nghiệm CIES
Chạy thử nghiệm cho các tổ hợp đại diện (hoặc toàn bộ 25 tổ hợp khi chạy trên GPU server/Kaggle).

In [ ]:
# 3. Run CIES Experiment Runner
# Dùng N_RUNS chuẩn từ src/config.py (mặc định 20, spec mục 7.1).
# KHÔNG hardcode số run thấp hơn spec — nếu cần giảm để debug/chạy local,
# phải hỏi lại người dùng trước khi quyết định (spec mục 7.2).
CURRENT_N_RUNS = N_RUNS

cies_all_results = []

# Mỗi tổ hợp (model, technique) chạy qua run_cies_experiment_isolated() —
# tự chạy trong 1 subprocess riêng, tránh torch (ANN) và xgboost cùng tồn
# tại trong 1 process (xung đột OpenMP runtime, xem src/utils/isolation.py).
for model_name in ['logistic_regression', 'random_forest', 'xgboost']:
    for technique in ['class_weighting', 'smote', 'smote_enn']:
        print(f'\n========================================')
        print(f'Running CIES: Model={model_name}, Technique={technique}')
        
        res = run_cies_experiment_isolated(
            model_name=model_name,
            imbalance_technique=technique,
            df_train=train_raw.sample(n=min(10000, len(train_raw)), random_state=SEED),
            df_test_fixed_eval=df_test_fixed_eval,
            target_col=TARGET_COL,
            n_runs=CURRENT_N_RUNS,
            feature_level=True,
        )
        cies_all_results.append(res)
        print(f'--> CIES Score: {res["cies_metrics"]["cies_score"]:.4f} (Spearman: {res["cies_metrics"]["mean_spearman"]:.4f})')

# Lưu kết quả CIES
save_cies_results(cies_all_results, RESULTS_DIR, filename='cies_summary_results.json')

--- 
## 4. Phân Tích Độ Ổn Định Giải Thích (CIES Analysis)
Tổng hợp kết quả bảng CIES score và vẽ Heatmap so sánh.

In [ ]:
# 4. Bảng tổng hợp CIES
summary_rows = []
for r in cies_all_results:
    summary_rows.append({
        'model': r['model_name'],
        'technique': r['imbalance_technique'],
        'cies_score': r['cies_metrics']['cies_score'],
        'mean_rank_distance': r['cies_metrics']['mean_rank_distance'],
        'std_rank_distance': r['cies_metrics']['std_rank_distance'],
        'mean_spearman': r['cies_metrics']['mean_spearman'],
    })

df_cies = pd.DataFrame(summary_rows)
print('=== BẢNG CIES STABILITY METRICS ===')
display(df_cies)

# Heatmap CIES
pivot_cies = df_cies.pivot(index='model', columns='technique', values='cies_score')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_cies, annot=True, cmap='viridis', fmt='.4f', vmin=0, vmax=1.0)
plt.title('Ma Trận CIES Score (Mô Hình × Kỹ Thuật Imbalance)', fontsize=13, fontweight='bold')
plt.ylabel('Mô Hình')
plt.xlabel('Kỹ Thuật Imbalance')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cies_heatmap.png', dpi=300)
plt.show()

--- 
## 5. Trade-off: Hiệu Năng Phân Loại (PR-AUC) vs Độ Tin Cậy Giải Thích (CIES)
Vẽ biểu đồ phân tán để tìm đường Pareto-optimal giữa PR-AUC và CIES Score.

In [ ]:
# 5. Trade-off Scatter Plot
# Ghép với kết quả PR-AUC từ notebook 03
benchmark_path = RESULTS_DIR / 'model_benchmark_results.csv'
if benchmark_path.exists():
    df_bench = pd.read_csv(benchmark_path)
    df_merged = pd.merge(df_cies, df_bench, on=['model', 'technique'] if 'technique' in df_bench.columns else None)
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df_merged,
        x='pr_auc',
        y='cies_score',
        hue='model',
        style='technique',
        s=150
    )
    plt.title('Trade-off: PR-AUC vs CIES Explanation Stability', fontsize=14, fontweight='bold')
    plt.xlabel('Hiệu Năng Phân Loại (PR-AUC)')
    plt.ylabel('Độ Tin Cậy Giải Thích (CIES Score)')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'tradeoff_prauc_vs_cies.png', dpi=300)
    plt.show()

--- 
## 6. Feature-Level CIES (Phân Tích Cấp Đặc Trưng)
Khảo sát các đặc trưng nào có rank ổn định nhất và đặc trưng nào bị biến động mạnh nhất giữa các lần chạy.

In [ ]:
# 6. Feature-level analysis cho tổ hợp tốt nhất
best_run = max(cies_all_results, key=lambda x: x['cies_metrics']['cies_score'])
print(f'Tổ hợp có CIES cao nhất: {best_run["combination"]}')

if 'feature_cies' in best_run and isinstance(best_run['feature_cies'], pd.DataFrame):
    df_feat = best_run['feature_cies']
    print('Top 10 features theo độ quan trọng SHAP trung bình:')
    display(df_feat.head(10))
    
    plt.figure(figsize=(10, 6))
    top_feats = df_feat.head(10)
    plt.errorbar(
        y=top_feats['feature'],
        x=top_feats['mean_abs_shap'],
        xerr=top_feats['shap_std'],
        fmt='o',
        color='darkblue',
        ecolor='red',
        elinewidth=2,
        capsize=5
    )
    plt.title(f'Độ Ổn Định Cấp Đặc Trưng - Mean |SHAP| ± Std ({best_run["combination"]})', fontsize=12, fontweight='bold')
    plt.xlabel('Mean |SHAP| Value')
    plt.ylabel('Feature')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'feature_level_stability.png', dpi=300)
    plt.show()

--- 
## 7. Hướng Dẫn Chạy Full Quy Mô (1000 Runs) Trên Kaggle / Colab
1. Upload toàn bộ thư mục `src/` lên Kaggle / Colab.
2. Đặt `N_RUNS = 20` trong `src/config.py`.
3. Sử dụng GPU/TPU accelerator để tăng tốc TreeSHAP và ANN.
4. Kết quả JSON từ `save_cies_results()` tải về để đưa vào báo cáo luận văn.